# YOLOv8 Vehicle Detection & Count
### AI Road Safety System â€” Kochi Traffic Analytics

This notebook demonstrates:
1. YOLOv8 model loading
2. Vehicle detection on traffic images/videos
3. Per-class vehicle counting (Car, Bus, Truck, Motorcycle, Bicycle)
4. Traffic density classification
5. Emergency / priority vehicle detection logic
6. Fine-tuning approach for custom Kerala traffic dataset


## 1. Install Dependencies

In [ ]:
import ultralytics, cv2, matplotlib
print('Ultralytics:', ultralytics.__version__)
print('OpenCV:', cv2.__version__)
print('Matplotlib:', matplotlib.__version__)


## 2. Load YOLOv8n Model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n model (downloads ~6 MB if not cached)
model = YOLO('yolov8n.pt')
model.info()

# Vehicle-relevant COCO class IDs
VEHICLE_CLASS_IDS = {1:'Bicycle', 2:'Car', 3:'Motorcycle', 5:'Bus', 7:'Truck'}
print('\nVehicle class mapping:')
for cid, lbl in VEHICLE_CLASS_IDS.items():
    print(f'  Class {cid:2d} -> {lbl}')


## 3. Vehicle Detection on a Single Image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request

# BGR color for each class
CLASS_COLORS = {1:(234,179,8), 2:(23,178,106), 3:(240,68,56), 5:(45,124,255), 7:(247,144,9)}

def detect_vehicles(image_path, confidence=0.40):
    img = cv2.imread(str(image_path))
    results = model(img, conf=confidence, verbose=False)[0]
    counts = {v:0 for v in VEHICLE_CLASS_IDS.values()}
    ann = img.copy()
    for box in results.boxes:
        cls_id = int(box.cls[0]); conf = float(box.conf[0])
        if cls_id not in VEHICLE_CLASS_IDS: continue
        label = VEHICLE_CLASS_IDS[cls_id]; color = CLASS_COLORS[cls_id]
        counts[label] += 1
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cv2.rectangle(ann,(x1,y1),(x2,y2),color,2)
        tag = f'{label} {conf:.2f}'
        (tw,th),_ = cv2.getTextSize(tag,cv2.FONT_HERSHEY_SIMPLEX,0.55,1)
        cv2.rectangle(ann,(x1,y1-th-6),(x1+tw+4,y1),color,-1)
        cv2.putText(ann,tag,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    return ann, counts, sum(counts.values())

# Download demo image
DEMO = Path('demo_traffic.jpg')
urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', DEMO)
print('Demo image ready:', DEMO)


In [ ]:
annotated, counts, total = detect_vehicles(DEMO)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
orig = cv2.cvtColor(cv2.imread(str(DEMO)), cv2.COLOR_BGR2RGB)
axes[0].imshow(orig); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
ann = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
axes[1].imshow(ann); axes[1].set_title(f'YOLOv8 â€” {total} vehicles', fontweight='bold'); axes[1].axis('off')
plt.tight_layout(); plt.savefig('detection_result.png', dpi=120); plt.show()

print('\nVehicle Count Breakdown:')
for label, cnt in counts.items():
    if cnt > 0:
        pct = round(cnt / max(total,1) * 100, 1)
        print(f'  {label:<12} {cnt:3d}  ({pct}%)')
print(f'  TOTAL        {total}')


## 4. Video Frame Analysis & Counting

In [ ]:
def detect_frame(frame, confidence=0.40):
    results = model(frame, conf=confidence, verbose=False)[0]
    counts = {v:0 for v in VEHICLE_CLASS_IDS.values()}
    ann = frame.copy()
    for box in results.boxes:
        cls_id = int(box.cls[0]); conf = float(box.conf[0])
        if cls_id not in VEHICLE_CLASS_IDS: continue
        label = VEHICLE_CLASS_IDS[cls_id]; color = CLASS_COLORS[cls_id]
        counts[label] += 1
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cv2.rectangle(ann,(x1,y1),(x2,y2),color,2)
        tag = f'{label} {conf:.2f}'
        (tw,th),_ = cv2.getTextSize(tag,cv2.FONT_HERSHEY_SIMPLEX,0.55,1)
        cv2.rectangle(ann,(x1,y1-th-6),(x1+tw+4,y1),color,-1)
        cv2.putText(ann,tag,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)
    return ann, counts, sum(counts.values())

def analyze_video(video_path, max_frames=20, confidence=0.40):
    cap = cv2.VideoCapture(str(video_path))
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, total_f // max_frames)
    print(f'Video: {total_f} frames @ {fps:.1f} FPS')
    print(f'Sampling every {step} frames ({max_frames} max samples)')
    agg = {v:0 for v in VEHICLE_CLASS_IDS.values()}
    totals = []; best = None; best_n = 0
    for i in range(max_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret: break
        ann, cnts, n = detect_frame(frame, confidence)
        for k, v in cnts.items(): agg[k] += v
        totals.append(n)
        if n >= best_n: best_n = n; best = ann
        print(f'  Frame {i+1:2d}: {n} vehicles')
    cap.release()
    return agg, totals, best

print('analyze_video() ready.')
print('Usage: agg, totals, best = analyze_video(r"C:/path/to/traffic.mp4")')


## 5. Vehicle Count Visualisation

In [ ]:
import matplotlib.pyplot as plt

# Demo data (replace with real results from detect_vehicles or analyze_video)
demo_counts = {'Bicycle':2, 'Car':45, 'Motorcycle':18, 'Bus':6, 'Truck':7}
demo_totals = [12,15,18,22,19,25,30,28,24,20,17,23]
CMAP = {'Bicycle':'#EAB308','Car':'#17B26A','Motorcycle':'#F04438','Bus':'#2D7CFF','Truck':'#F79009'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('YOLOv8 Vehicle Detection â€” Kochi Traffic', fontsize=15, fontweight='bold')

labels = list(demo_counts.keys()); values = list(demo_counts.values())
colors = [CMAP[l] for l in labels]

bars = axes[0].bar(labels, values, color=colors, edgecolor='white')
axes[0].set_title('Count by Vehicle Type', fontweight='bold')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(val), ha='center', fontweight='bold')

total = sum(values)
axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title(f'Share per Type (Total: {total})', fontweight='bold')

x = range(1, len(demo_totals)+1)
axes[2].plot(x, demo_totals, color='#2D7CFF', linewidth=2.5, marker='o', markersize=5)
axes[2].fill_between(x, demo_totals, alpha=0.15, color='#2D7CFF')
axes[2].set_title('Vehicles per Sampled Frame', fontweight='bold')
axes[2].set_xlabel('Frame Index'); axes[2].set_ylabel('Count'); axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('vehicle_count_analysis.png', dpi=120); plt.show()
print('Saved: vehicle_count_analysis.png')


## 6. Traffic Density & Priority Classification

In [ ]:
def classify_density(total, frames):
    avg = total / max(frames, 1)
    if avg >= 8:   return 'High',   'Signal extension + congestion management'
    elif avg >= 4: return 'Medium', 'Standard adaptive signal timing'
    else:          return 'Low',    'Normal flow â€” no action needed'

def priority_decision(counts, density, emergency=False):
    bus = counts.get('Bus', 0); truck = counts.get('Truck', 0)
    if emergency or bus >= 3:
        return {'priority':'HIGH â€” Bus Priority',    'action':'Extend green phase +25 sec',    'color':'RED'}
    elif density == 'High':
        return {'priority':'MEDIUM â€” Congestion',    'action':'Adaptive timing + wardens',      'color':'ORANGE'}
    else:
        return {'priority':'NORMAL â€” Standard Flow', 'action':'Standard signal cycle',           'color':'GREEN'}

total_v = sum(demo_counts.values())
density, msg = classify_density(total_v, len(demo_totals))
dec = priority_decision(demo_counts, density)

print('=' * 50)
print('  TRAFFIC ANALYSIS REPORT')
print('=' * 50)
print(f'  Total Vehicles : {total_v}')
print(f'  Avg per Frame  : {total_v/len(demo_totals):.1f}')
print(f'  Traffic Density: {density}')
print(f'  Notes          : {msg}')
print('-' * 50)
print(f'  Priority Level : {dec["priority"]}')
print(f'  Recommended Act: {dec["action"]}')
print('=' * 50)


## 7. Fine-tuning YOLOv8 on Custom Dataset (Concept)

To train on a **custom Kerala traffic dataset**:

1. Annotate traffic images using Roboflow or LabelImg
2. Export in YOLO format (`images/` + `labels/` folders)
3. Create `dataset.yaml`
4. Run `model.train()`

> Do NOT run the training cell without a labelled dataset.


In [ ]:
YAML = '''
# Kerala Traffic Dataset
path: /path/to/dataset
train: images/train
val: images/val
nc: 5
names: [bicycle, car, motorcycle, bus, truck]
'''
print('Dataset YAML structure:')
print(YAML)

TRAIN = '''
from ultralytics import YOLO
model = YOLO("yolov8n.pt")  # start from COCO pretrained weights
results = model.train(
    data="dataset.yaml",
    epochs=50, imgsz=640, batch=16,
    device=0,  # 0=GPU, "cpu"=CPU
    project="kerala_traffic"
)
metrics = model.val()
print("mAP50:", metrics.box.map50)
'''
print('Training code preview:')
print(TRAIN)


## 8. Model Evaluation Metrics

| Metric | Formula | Meaning |
|--------|---------|--------|
| Precision | TP/(TP+FP) | Correct detections / All detections |
| Recall | TP/(TP+FN) | Detected / All actual vehicles |
| mAP50 | Mean AP at IoU=0.50 | Standard detection accuracy |
| mAP50-95 | Multi-threshold | Strict accuracy metric |

### YOLOv8n COCO baseline:
| Class | mAP50 |
|-------|-------|
| Car | 0.74 |
| Bus | 0.71 |
| Truck | 0.68 |
| Motorcycle | 0.56 |
| Bicycle | 0.52 |


In [ ]:
import matplotlib.pyplot as plt

classes = ['Car','Bus','Truck','Motorcycle','Bicycle']
map50   = [0.74, 0.71, 0.68, 0.56, 0.52]
colors  = ['#17B26A','#2D7CFF','#F79009','#F04438','#EAB308']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(classes, map50, color=colors, edgecolor='white', height=0.5)
ax.set_xlim(0, 1); ax.set_xlabel('mAP@50', fontsize=12)
ax.set_title('YOLOv8n Vehicle Detection Performance', fontsize=14, fontweight='bold')
for bar, val in zip(bars, map50):
    ax.text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.2f}', va='center', fontweight='bold')
ax.axvline(0.5, color='#F04438', linewidth=1.5, linestyle='--', label='0.50 threshold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('map_scores.png', dpi=120); plt.show()


## 9. Flask API Integration

```
Browser --> POST /api/yolo/detect --> Flask
               (video upload)           |
                                   OpenCV reads video
                                        |
                                   YOLOv8n inference (20 sampled frames)
                                        |
                                   Count per class + annotated frame
                                        |
Browser <-- JSON (counts, frame_b64, density, emergency, priority)
```

### Emergency Priority Heuristic
```python
# If large bus/truck detected (area > 4% of frame)
if cls_id in (5, 7):  # Bus or Truck
    area = (x2 - x1) * (y2 - y1)
    if area > 0.04 * frame_h * frame_w:
        emergency_detected = True
        # Dashboard shows red alert + recommends green-phase extension
```


In [ ]:
print('Notebook complete!')
print('Covered: model load, detection, counting, density, priority, fine-tuning, metrics, API')
print('Output files: detection_result.png, vehicle_count_analysis.png, map_scores.png')
